# Regresi Linear

Estimasi waktu: **45 menit**

Kita berperan sebagai konsultan untuk sebuah perusahaan otomotif. Pertanyaannya: faktor apa saja yang menaikkan harga jual mobil, dan seberapa akurat harga itu bisa diprediksi dari spesifikasinya?

Notebook ini menjalankan regresi linear sederhana pada data harga mobil, memeriksa asumsi yang harus dipenuhi, lalu mencoba beberapa cara transformasi data.

## Tujuan

- memilih fitur yang berpengaruh terhadap harga;
- memeriksa asumsi regresi linear;
- menerapkan transformasi pada target yang miring;
- mengevaluasi model dengan MSE dan R kuadrat.

## Persiapan

Pustaka yang dipakai: `pandas`, `numpy`, `seaborn`, `matplotlib`, `scipy`, dan `scikit-learn`.

In [ ]:
import warnings
from urllib.request import urlretrieve
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## Membaca data

Data penjualan mobil diambil dari repositori publik. Satu baris berisi satu model mobil beserta spesifikasi teknis dan harga jualnya.

In [ ]:
data_path = Path("data/CarPrice_Assignment.csv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/CarPrice_Assignment.csv"

if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(data_url, data_path)

cars = pd.read_csv(data_path)
cars.head()

In [ ]:
cars.info()

Dataset berisi 205 baris dan 26 kolom. Kolom `price` adalah target, sisanya calon prediktor. Tipe datanya campuran: numerik dan teks.

In [ ]:
cars.describe()

## Pembersihan dan penyiapan data

Pertama, cek nilai kosong dan baris duplikat.

In [ ]:
print("Total nilai kosong:", cars.isnull().sum().sum())
print("Duplikat car_ID:", cars["car_ID"].duplicated().sum())

Kolom `CarName` memuat merek dan model sekaligus. Untuk pemodelan, cukup mereknya saja. Beberapa merek juga salah ketik, jadi perlu diseragamkan.

In [ ]:
cars["brand"] = cars["CarName"].str.split(" ").str[0].str.lower()

typo_map = {
    "vw": "volkswagen",
    "vokswagen": "volkswagen",
    "maxda": "mazda",
    "porcshce": "porsche",
    "toyouta": "toyota",
}
cars["brand"] = cars["brand"].replace(typo_map)
cars["brand"].unique()

In [ ]:
plt.figure(figsize=(14, 5))
sns.countplot(x=cars["brand"], order=cars["brand"].value_counts().index)
plt.xlabel("Merek")
plt.ylabel("Jumlah mobil")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
cars.drop(columns=["car_ID", "symboling", "CarName"], inplace=True)
cars.info()

### Rekayasa fitur

Rata-rata harga tiap merek dipakai untuk mengelompokkan mobil menjadi tiga kelas: Budget, Mid_Range, dan Luxury.

In [ ]:
brand_price = (
    cars.groupby("brand", as_index=False)["price"]
    .mean()
    .rename(columns={"price": "brand_avg_price"})
)

cars = cars.merge(brand_price, on="brand")
cars["brand_avg_price"].describe()

In [ ]:
cars["brand_category"] = pd.cut(
    cars["brand_avg_price"],
    bins=[0, 10000, 20000, np.inf],
    labels=["Budget", "Mid_Range", "Luxury"],
    right=False,
)
cars["brand_category"].value_counts()

## Analisis data eksploratif

Variabel kategorikal yang tersedia: `brand_category`, `fueltype`, `enginetype`, `carbody`, `doornumber`, `enginelocation`, `fuelsystem`, `cylindernumber`, `aspiration`, dan `drivewheel`.

Boxplot berikut memperlihatkan sebaran harga pada tiap kategori, termasuk kemungkinan pencilan.

In [ ]:
categorical_features = [
    "fueltype", "aspiration", "carbody", "drivewheel", "enginetype", "brand_category",
]

fig, axes = plt.subplots(3, 2, figsize=(12, 14))
for column, ax in zip(categorical_features, axes.ravel()):
    sns.boxplot(x=cars[column], y=cars["price"], ax=ax)
    ax.set_title(column)
    ax.set_xlabel("")
fig.tight_layout()
plt.show()

Selanjutnya, koefisien korelasi Pearson antara setiap fitur numerik dan harga.

In [ ]:
price_correlation = cars.corr(numeric_only=True)["price"].sort_values(ascending=False)
price_correlation

In [ ]:
sns.pairplot(cars[["price", "enginesize", "horsepower", "curbweight", "citympg"]])
plt.show()

## Menguji asumsi regresi linear

Regresi linear mengasumsikan hubungan target dan prediktor bersifat linear, varians residual konstan (*homoskedastisitas*), target berdistribusi normal, dan prediktor tidak saling berkorelasi kuat (*multikolinearitas*).

### 1. Linearitas

Hubungan diperiksa lewat scatter plot dengan garis regresi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for column, ax in zip(["enginesize", "horsepower"], axes):
    sns.regplot(x=cars[column], y=cars["price"], ax=ax, scatter_kws={"s": 15})
    ax.set_title(column)
fig.tight_layout()
plt.show()

In [ ]:
sns.regplot(x=cars["curbweight"], y=cars["price"], scatter_kws={"s": 15})
plt.show()

### 2. Homoskedastisitas

Residual diplot terhadap `enginesize`. Pola melebar seperti corong menandakan varians error tidak seragam, sehingga asumsi ini kemungkinan tidak terpenuhi.

In [ ]:
plt.figure(figsize=(8, 5))
sns.residplot(x=cars["enginesize"], y=cars["price"], scatter_kws={"s": 15})
plt.show()

### 3. Normalitas target

Tiga grafik sekaligus: histogram, boxplot, dan Q-Q plot.

In [ ]:
def plot_normality(values, label):
    fig, (ax_hist, ax_box, ax_qq) = plt.subplots(1, 3, figsize=(15, 4))
    ax_hist.hist(values, bins=30, edgecolor="white")
    ax_hist.set_title(f"Histogram {label}")
    ax_box.boxplot(values, vert=False)
    ax_box.set_title(f"Boxplot {label}")
    stats.probplot(values, dist="norm", plot=ax_qq)
    ax_qq.set_title(f"Q-Q {label}")
    fig.tight_layout()
    plt.show()


plot_normality(cars["price"], "price")

Harga belum normal: distribusinya miring ke kanan, sebagian besar mobil berharga di bawah 15.000, dan ada pencilan jauh di ekor kanan. Uji statistik memperkuat hal ini.

In [ ]:
stats.normaltest(cars["price"].to_numpy())

## Transformasi target

Salinan data sebelum transformasi disimpan agar bisa dibandingkan nanti.

In [ ]:
raw_cars = cars.copy()

### Transformasi logaritma

`np.log` menarik ekor kanan ke tengah sehingga distribusi lebih simetris. Untuk nilai kecil, `np.log1p` lebih akurat karena aman terhadap nol.

In [ ]:
cars["price"] = np.log(cars["price"])
plot_normality(cars["price"], "log price")
stats.normaltest(cars["price"].to_numpy())

Hasilnya jauh lebih dekat ke kurva lonceng, meski belum sempurna.

### Latihan: transformasi Box-Cox

Box-Cox mencari pangkat terbaik secara otomatis. Terapkan pada data asli (`raw_cars`) dan uji kenormalannya.

In [ ]:
boxcox_prices, boxcox_lambda = stats.boxcox(raw_cars["price"])
print("lambda:", boxcox_lambda)
stats.normaltest(boxcox_prices)

p-value Box-Cox masih kecil, jadi distribusi harga mobil ini tetap belum normal meski sudah ditransformasi.

### 4. Multikolinearitas

Heatmap korelasi antar fitur numerik membantu melihat pasangan prediktor yang saling terkait kuat.

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(cars.corr(numeric_only=True), annot=True, fmt=".2f", cmap="YlGnBu")
plt.show()

Multikolinearitas masih muncul pada beberapa pasangan fitur, misalnya ukuran mesin dengan tenaga kuda. Untuk sementara fitur dibiarkan, dan penanganannya diserahkan ke model regularisasi pada materi berikutnya.

## Model regresi linear

Fitur yang dipilih berdasarkan hasil eksplorasi:

**Numerik:** curbweight, carlength, carwidth, enginesize, boreratio, horsepower, wheelbase, citympg, highwaympg.

**Kategorikal:** enginetype, fueltype, carbody, aspiration, cylindernumber, drivewheel, brand_category.

In [ ]:
selected_columns = [
    "price", "fueltype", "aspiration", "carbody", "drivewheel", "wheelbase",
    "brand_category", "curbweight", "enginetype", "cylindernumber", "enginesize",
    "boreratio", "horsepower", "carlength", "carwidth", "citympg", "highwaympg",
]

selected = cars[selected_columns].copy()
categorical_columns = selected.select_dtypes(exclude="number").columns.tolist()
numeric_columns = [column for column in selected_columns if column not in categorical_columns + ["price"]]

print("Kategorikal:", categorical_columns)
print("Numerik:", numeric_columns)

Beberapa kategori hanya muncul sekali, misalnya `three` dan `twelve` pada `cylindernumber`. Karena itu data dibagi lebih dulu sebelum encoding, supaya kategori langka di data uji tidak memicu error.

In [ ]:
for column in categorical_columns:
    print(column, "->", selected[column].nunique(), "kategori")

### One-hot encoding

`ColumnTransformer` memungkinkan setiap kelompok kolom ditransformasi secara terpisah. Kolom kategorikal di-encode, kolom numerik dilewatkan apa adanya.

In [ ]:
features = selected.drop(columns="price")
target = selected["price"]

encoder = ColumnTransformer(
    transformers=[("one_hot", OneHotEncoder(sparse_output=False), categorical_columns)],
    remainder="passthrough",
)

features_encoded = encoder.fit_transform(features)
feature_names = [name.split("__", 1)[-1] for name in encoder.get_feature_names_out()]
encoded_frame = pd.DataFrame(features_encoded, columns=feature_names)
encoded_frame.head()

### Pembagian data dan standardisasi

30% data dipakai sebagai data uji. `StandardScaler` di-fit hanya pada data latih, lalu diterapkan ke data uji.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    encoded_frame, target, test_size=0.30, random_state=0
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
predictions[:10]

### Evaluasi model

MSE mengukur rata-rata kuadrat selisih prediksi dan nilai aktual. RMSE adalah akar MSE sehingga satuannya sama dengan target. R kuadrat menunjukkan proporsi variasi target yang bisa dijelaskan model.

In [ ]:
mse = mean_squared_error(y_test, predictions)
print("MSE :", mse)
print("RMSE:", np.sqrt(mse))
print("R2  :", r2_score(y_test, predictions))

R kuadrat yang negatif menandakan model lebih buruk daripada sekadar menebak rata-rata, tanda overfitting pada data latih.

### Pipeline

Pipeline merangkai transformasi dan model dalam satu objek, sehingga urutan langkah tidak bisa tertukar.

In [ ]:
pipeline = Pipeline(steps=[("scaler", StandardScaler()), ("model", LinearRegression())])
pipeline.fit(X_train, y_train)

pipeline_predictions = pipeline.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, pipeline_predictions)))
print("R2  :", r2_score(y_test, pipeline_predictions))

### Latihan: pipeline lengkap

Gabungkan encoding, standardisasi, dan regresi ke dalam satu pipeline, latih pada seluruh data, lalu hitung R kuadratnya.

In [ ]:
full_pipeline = Pipeline(steps=[
    ("one_hot", ColumnTransformer(
        transformers=[("one_hot", OneHotEncoder(sparse_output=False), categorical_columns)],
        remainder="passthrough",
    )),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

full_pipeline.fit(features, target)
full_predictions = full_pipeline.predict(features)
r2_score(target, full_predictions)

## Ringkasan

Harga mobil pada dataset ini miring ke kanan dan belum normal meski sudah ditransformasi. Regresi linear pada fitur terpilih masih menyisakan multikolinearitas, sehingga model regularisasi menjadi langkah lanjutan yang wajar.